# Research Phase: Direct HuggingFace Transformers
*Maximum Control & Experimentation*

This notebook demonstrates the most direct way to work with HuggingFace models using the transformers library. Perfect for research, experimentation, and when you need full control over model behavior.

## When to Use This Approach
- Research and experimentation
- Maximum performance requirements
- Full control over model parameters
- Custom model architectures
- Learning how transformers work internally

## Prerequisites
- transformers library
- torch
- Basic understanding of PyTorch

## Setup

In [14]:
import torch
from transformers import (
    pipeline, 
    AutoTokenizer, 
    AutoModelForCausalLM,
    AutoModelForSequenceClassification
)
import time
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Direct Model Loading & Inference

In [2]:
# Load model and tokenizer directly
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Add padding token for GPT-2
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name, output_attentions=True)

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model: {model_name}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device: {device}")

In [3]:
# Direct inference - maximum control
def generate_text(prompt, max_length=50, temperature=0.7, top_p=0.9):
    """
    Generate text with full control over parameters
    """
    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    # Generate with custom parameters
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            num_return_sequences=1
        )
    
    # Decode output
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Remove prompt from output
    if generated_text.startswith(prompt):
        generated_text = generated_text[len(prompt):].strip()
    
    return generated_text

# Test direct generation
prompt = "The future of AI is"
result = generate_text(prompt, max_length=30)
print(f"Prompt: {prompt}")
print(f"Generated: {result}")

## Pipeline Approach (Still Direct)

In [4]:
# Pipeline - still direct but more convenient
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device
)

# Custom generation parameters
def generate_with_pipeline(prompt, **kwargs):
    """
    Generate using pipeline with custom parameters
    """
    outputs = generator(
        prompt,
        max_length=50,
        num_return_sequences=1,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        **kwargs
    )
    return outputs[0]['generated_text']

# Test pipeline generation
prompt = "In quantum computing,"
result = generate_with_pipeline(prompt)
print(f"Prompt: {prompt}")
print(f"Generated: {result}")

## Model Inspection & Analysis

In [9]:
# Inspect model architecture
print("Model Architecture:")
print(model)

print(f"\nConfig: {model.config}")

# Look at model layers
print(f"\nNumber of layers: {model.config.n_layer}")
print(f"Hidden size: {model.config.n_embd}")
print(f"Vocab size: {model.config.vocab_size}")

# Examine attention patterns (for research)
def analyze_attention(text):
    """
    Analyze attention patterns in the model
    Note: GPT-2 doesn't output attentions by default
    """
    print("🔍 Attention Analysis")
    print("Note: GPT-2 doesn't support attention output by default.")
    print("For attention analysis, consider using BERT or other encoder models.")
    print()
    
    # Show tokenization instead
    inputs = tokenizer(text, return_tensors="pt").to(device)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    print(f"Input text: '{text}'")
    print(f"Tokenized: {tokens}")
    print(f"Token IDs: {inputs['input_ids'][0].tolist()}")
    print()
    
    # Show model forward pass (without attention)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
    
    print(f"Output logits shape: {logits.shape}")
    print(f"Vocabulary size: {logits.shape[-1]}")
    
    # Get top predictions for next token
    next_token_logits = logits[0, -1, :]  # Last position
    top_tokens = torch.topk(next_token_logits, k=5)
    
    print("Top 5 predicted next tokens:")
    for idx, score in zip(top_tokens.indices, top_tokens.values):
        token = tokenizer.convert_ids_to_tokens([idx.item()])[0]
        print(".3f")
    
    return None  # No attentions available

# Analyze attention on a sample text
sample_text = "The cat sat on the mat"
analyze_attention(sample_text)

## Performance Benchmarking

In [10]:
def benchmark_generation(prompts, num_runs=5):
    """
    Benchmark generation performance
    """
    results = []
    
    for prompt in prompts:
        times = []
        
        for _ in range(num_runs):
            start_time = time.time()
            _ = generate_text(prompt, max_length=30)
            times.append(time.time() - start_time)
        
        avg_time = sum(times) / len(times)
        results.append({
            'prompt': prompt,
            'avg_time': avg_time,
            'tokens_generated': 30 - len(tokenizer.encode(prompt))
        })
    
    return results

# Benchmark different prompts
test_prompts = [
    "Hello",
    "The weather today is",
    "Machine learning is"
]

benchmark_results = benchmark_generation(test_prompts)
print("Benchmark Results:")
for result in benchmark_results:
    print(".3f")

## Custom Model Modifications

In [11]:
# Example: Modify model behavior (research purposes)
def create_modified_model(original_model):
    """
    Create a modified version of the model for experimentation
    """
    # Freeze most layers (keep only last layer trainable)
    for name, param in original_model.named_parameters():
        if 'transformer.h' in name and not name.endswith('.ln_f.weight') and not name.endswith('.ln_f.bias'):
            # Freeze all transformer layers except layer norms
            if not any(layer in name for layer in ['11.ln_f', '10.ln_f', '9.ln_f']):  # Keep last 3 layers
                param.requires_grad = False
        elif 'lm_head' in name:
            # Keep output layer trainable
            param.requires_grad = True
    
    trainable_params = sum(p.numel() for p in original_model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in original_model.parameters())
    
    print(f"Trainable parameters: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")
    print(f"Frozen parameters: {total_params - trainable_params:,}")
    
    return original_model

# Create modified model
modified_model = create_modified_model(model)

# Test that it still works
test_prompt = "AI will"
result = generate_text(test_prompt, max_length=25)
print(f"Modified model test: '{test_prompt}' → '{result}'")

## Research-Phase Best Practices

In [15]:
RESEARCH_BEST_PRACTICES = {
    "experimentation": [
        "Use direct model access for parameter exploration",
        "Experiment with different generation strategies",
        "Analyze attention patterns and model internals",
        "Test edge cases and failure modes"
    ],
    "performance": [
        "Profile memory usage and inference speed",
        "Compare different model sizes and architectures",
        "Test quantization effects on quality",
        "Benchmark on your specific use cases"
    ],
    "debugging": [
        "Use torch.no_grad() for inference speed",
        "Monitor gradient flow during training",
        "Check tokenization consistency",
        "Validate model outputs systematically"
    ],
    "scaling": [
        "Start with small models for experimentation",
        "Use mixed precision for faster training",
        "Implement gradient checkpointing for large models",
        "Consider model parallelism for very large models"
    ]
}

print("🔬 Research Phase Best Practices:")
for category, practices in RESEARCH_BEST_PRACTICES.items():
    print(f"\n{category.upper()}:")
    for practice in practices:
        print(f"  ✓ {practice}")